In [ ]:
!nvidia-smi

Thu Nov  5 10:24:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Access your specific "project" folder
folder_path = '/content/drive/MyDrive/project'

import os
# Check if the folder exists and list its contents
if os.path.exists(folder_path):
    print("Folder found! Contents:", os.listdir(folder_path))
else:
    print(f"Directory '{folder_path}' not found. Check if the folder name is different or located in 'Shared with me'.")

Mounted at /content/drive
Folder found! Contents: ['Construction and Demolition Waste Object Detection Dataset  (CODD).zip', 'ai-service-colab.zip']


In [ ]:
!unzip -q '/content/drive/MyDrive/project/ai-service-colab.zip' -d /content/
%cd /content/ai-service/training
!pip install ultralytics pyyaml

replace /content/ai-service/detector.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: /content/ai-service/training


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

zip_path = '/content/drive/MyDrive/project/ai-service-colab.zip'
if os.path.exists(zip_path):
    print("Found zip file. Extracting...")
    !unzip -q "{zip_path}" -d /content/
    print("Extraction complete. Contents of /content:")
    print(os.listdir('/content'))
else:
    print(f"Could not find zip file at: {zip_path}. Please check your Google Drive folder structure.")

Mounting Google Drive...
Mounted at /content/drive
Found zip file. Extracting...
replace /content/ai-service/detector.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: Extraction complete. Contents of /content:
['.config', 'drive', 'ai-service', 'sample_data']


In [ ]:
%cd /content/ai-service/training
!pip install -r requirements.txt

/content/ai-service/training
  Using cached roboflow-1.7.3-py3-none-any.whl.metadata (12 kB)
  Using cached pymongo-4.19.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
Using cached roboflow-1.7.3-py3-none-any.whl (381 kB)
Using cached typer-0.25.1-py3-none-any.whl (58 kB)
Using cached pymongo-4.19.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (1.1 MB)
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [roboflow]


In [ ]:
import os

codd_zip_path = '/content/drive/MyDrive/project/Construction and Demolition Waste Object Detection Dataset  (CODD).zip'
target_extract_dir = '/content/dataset'

if os.path.exists(codd_zip_path):
    print("Found CODD dataset zip file. Extracting...")
    os.makedirs(target_extract_dir, exist_ok=True)
    !unzip -q "{codd_zip_path}" -d "{target_extract_dir}"
    print(f"Extraction complete. Dataset extracted to {target_extract_dir}")
    print("Contents:", os.listdir(target_extract_dir)[:5])
else:
    print(f"Could not find the dataset at {codd_zip_path}. Please verify the filename in Google Drive.")

Found CODD dataset zip file. Extracting...
Extraction complete. Dataset extracted to /content/dataset
Contents: ['Construction and Demolition Waste Object Detection Dataset  (CODD)']


In [ ]:
!mkdir -p data/codd
!unzip -q "/content/drive/MyDrive/project/Construction and Demolition Waste Object Detection Dataset  (CODD).zip" -d data/codd
!python prepare_codd.py data/codd data/codd_yolo

labels found in source: {'wood': 2055, 'plastic': 1247, 'general_w': 1296, 'foam': 1341, 'brick': 2093, 'concrete': 2155, 'tile': 2065, 'pipes': 1231, 'gypsum_board': 1777, 'stone': 1259}
kept as unified classes: {'wood': 2055, 'brick': 2093, 'concrete': 2155, 'tiles': 2065}


In [ ]:
!python prepare_codd.py "/content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)" data/codd_yolo

labels found in source: {'wood': 2055, 'plastic': 1247, 'general_w': 1296, 'foam': 1341, 'brick': 2093, 'concrete': 2155, 'tile': 2065, 'pipes': 1231, 'gypsum_board': 1777, 'stone': 1259}
kept as unified classes: {'wood': 2055, 'brick': 2093, 'concrete': 2155, 'tiles': 2065}


In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

Roboflow API key: ··········


In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
datasets = [
    ("iaac", "constrcution-materials", 2),
    ("enzo-guo-jfodj", "cd_obb", None), # Auto-detect version if 1 fails
    ("conapi", "material_database-i52bc", None) # Auto-detect version if 1 fails
]

for ws, proj, ver in datasets:
    print(f"Accessing {ws}/{proj}...")
    project = rf.workspace(ws).project(proj)
    if ver is None:
        # Fetch the latest available version automatically
        versions = project.versions()
        if not versions:
            print(f"Skipping {ws}/{proj}: No versions found.")
            continue
        ver = int(versions[0].version.split('-')[-1]) if '-' in versions[0].version else 1

    print(f"Downloading {ws}/{proj} version {ver}...")
    try:
        project.version(ver).download("yolov8", location=f"data/raw/{ws}__{proj}")
    except Exception as e:
        print(f"Failed downloading version {ver}: {e}. Trying first available instead.")
        first_ver = int(project.versions()[0].version)
        project.version(first_ver).download("yolov8", location=f"data/raw/{ws}__{proj}")

Accessing iaac/constrcution-materials...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/iaac__constrcution-materials in yolov8:: 100%|██████████| 866/866 [00:00<00:00, 8054.47it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Accessing enzo-guo-jfodj/cd_obb...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/enzo-guo-jfodj__cd_obb in yolov8:: 100%|██████████| 3671/3671 [00:00<00:00, 9837.63it/s]

Accessing conapi/material_database-i52bc...
loading Roboflow workspace...


loading Roboflow project...
Skipping conapi/material_database-i52bc: No versions found.


In [ ]:
!python merge_yolo.py data/unified data/codd_yolo data/raw/iaac__constrcution-materials data/raw/enzo-guo-jfodj__cd_obb

images per split: {'train': 2687, 'val': 587, 'test': 357}
boxes per class: {'wood': 6549, 'brick': 6400, 'concrete': 4634, 'tiles': 2014}


In [ ]:
new_common_content = """\"\"\"Shared class mapping: any dataset label -> the unified classes.\"\"\"
UNIFIED = ["concrete", "brick", "wood", "tiles"]
KEYWORDS = {
    "concrete": ["concrete"],
    "brick": ["brick"],
    "wood": ["wood", "timber", "lumber"],
    "tiles": ["tile"]
}

def map_name(name: str):
    \"\"\"Return unified class if EXACTLY one matches, else None (drops ambiguous labels like 'brickconcrete').\"\"\"
    n = name.lower().replace("_", " ").replace("-", " ")
    hits = {u for u, ks in KEYWORDS.items() if any(k in n for k in ks)}
    return next(iter(hits)) if len(hits) == 1 else None
"""

with open('common.py', 'w') as f:
    f.write(new_common_content)

print('Successfully updated common.py to remove empty classes (steel and glass).')

Successfully updated common.py to remove empty classes (steel and glass).


In [ ]:
!python merge_yolo.py data/unified data/codd_yolo data/raw/iaac__constrcution-materials data/raw/enzo-guo-jfodj__cd_obb

images per split: {'train': 2687, 'val': 587, 'test': 357}
boxes per class: {'wood': 6549, 'brick': 6400, 'concrete': 4634, 'tiles': 2014}


In [ ]:
!python train_yolo.py data/unified/data.yaml materials_v3 --model yolov8s.pt --epochs 50 --imgsz 640 --batch 16 --patience 10

training boxes per class: {'concrete': 3363, 'brick': 4706, 'wood': 4826, 'tiles': 1621}
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/unified/data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, m

In [ ]:
with open('train_yolo.py', 'r') as f:
    print(f.read())

In [ ]:
import os

print("Searching for all saved weights (.pt files)...")
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.pt'):
            print(os.path.join(root, file))

In [ ]:
import os
import shutil

# Source directories to search
search_dir = '/content'
# Destination directory in your Google Drive
drive_dest_dir = '/content/drive/MyDrive/project/weights'
os.makedirs(drive_dest_dir, exist_ok=True)

print("Searching and copying weights (.pt) to Google Drive...")
found_any = False
for root, dirs, files in os.walk(search_dir):
    # Skip Google Drive folders during local search to speed up and avoid recursive copying
    if 'drive' in root.split(os.sep):
        continue
    for file in files:
        if file.endswith('.pt'):
            src_path = os.path.join(root, file)
            dest_path = os.path.join(drive_dest_dir, f"{os.path.basename(os.path.dirname(src_path))}_{file}")
            try:
                shutil.copy2(src_path, dest_path)
                print(f"Copied: {src_path} -> {dest_path}")
                found_any = True
            except Exception as e:
                print(f"Failed to copy {src_path}: {e}")

if not found_any:
    print("No .pt files found in local directories. Please ensure training finished successfully.")
else:
    print(f"\nAll weights successfully copied to Google Drive! You can find them in: {drive_dest_dir}")

Searching and copying weights (.pt) to Google Drive...
Copied: /content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)/PyTorch Models/640 x 640/YoloV8l.pt -> /content/drive/MyDrive/project/weights/640 x 640_YoloV8l.pt
Copied: /content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)/PyTorch Models/640 x 640/YoloV8l-seg.pt -> /content/drive/MyDrive/project/weights/640 x 640_YoloV8l-seg.pt
Copied: /content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)/PyTorch Models/640 x 640/YoloV8s.pt -> /content/drive/MyDrive/project/weights/640 x 640_YoloV8s.pt
Copied: /content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)/PyTorch Models/640 x 640/YoloV8n.pt -> /content/drive/MyDrive/project/weights/640 x 640_YoloV8n.pt
Copied: /content/dataset/Construction and Demolition Waste Object Detection Dataset  (CODD)/PyTorch Models/640 x 640/YoloV8n-seg.pt -> /content/drive/MyDrive/project/weights